In [1]:
import asyncio
import os
from typing import Annotated
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg_pool import AsyncConnectionPool
from typing_extensions import TypedDict
from langgraph.prebuilt import ToolNode, tools_condition


/home/toqeer-yasir/miniconda3/envs/lg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
DATABASE_URL = os.getenv("DATABASE_URL")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [4]:
model = ChatOpenRouter(
    model="openrouter/free",
    api_key=OPENROUTER_API_KEY,
    temperature=0.7,
)

In [5]:
from tools import get_tools
tools = await get_tools()
print(len(tools))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1476.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully connected to database heaving 18 chunks.
15


In [6]:
tool_node = ToolNode(tools= tools)

In [7]:
model_with_tools = model.bind_tools(tools= tools)

In [8]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [9]:
async def chatnode(state: ChatState):

    response = await model_with_tools.ainvoke(state["messages"])

    return {
        "messages": [response]
    }

In [10]:
builder = StateGraph(ChatState)

In [11]:
builder.add_node("chatnode", chatnode)
builder.add_node("tools", tool_node)

In [12]:
builder.add_edge(START, "chatnode")
builder.add_conditional_edges("chatnode", tools_condition)
builder.add_edge("tools", "chatnode")

In [13]:
async def main():

    async with AsyncConnectionPool(
        DATABASE_URL,
        min_size=4,
        max_size=10
    ) as pool:

        checkpointer = AsyncPostgresSaver(pool)
        await checkpointer.setup()

        graph = builder.compile(checkpointer=checkpointer)

        thread_id = "chat_244"
        user_id = "user_244"

        print("\nType 'exit' to quit.\n")

        while True:

            query = input("You : ")

            if query.lower() == "exit":
                break

            result = await graph.ainvoke({"messages": [HumanMessage(content= query)]},
                config={"configurable": {"thread_id": thread_id,}},
                context={"user_id": user_id})

            result = "\nAssistant :", result["messages"][-1].content
            display(Markdown(f"{result[0]} {result[1]}"))

In [14]:
from IPython.display import display, Markdown

In [15]:
await main()


Type 'exit' to quit.



TooManyRequestsResponseError: Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day